# Kémzy àvátâr — PersonaLive CUDA Render Proof

This notebook runs the **real Kémzy PersonaLive backend** on a Kaggle GPU. It keeps the large model weights in the Kaggle session and does not download them to the phone.

Target: CUDA → PersonaLive → reference fusion → 4 driving frames → generated neural frame.

In [ ]:
!nvidia-smi
import torch
print('torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM GiB:', round(torch.cuda.get_device_properties(0).total_memory/1024**3, 2))

In [ ]:
!rm -rf /kaggle/working/Kemzy-LiveAvatar /kaggle/working/PersonaLive
!git clone --depth 1 --branch feature/backend-render-gateway https://github.com/eneokonaniebiet/Kemzy-LiveAvatar.git /kaggle/working/Kemzy-LiveAvatar
!git clone --depth 1 --branch abdd112e01dcf7d89122c2e5efa29fcff0669740 https://github.com/GVCLab/PersonaLive.git /kaggle/working/PersonaLive
%cd /kaggle/working/PersonaLive
!pip install -q -r requirements_base.txt

In [ ]:
# Download the official PersonaLive weights into Kaggle only.
# This is intentionally NOT a phone/Android download.
!python tools/download_weights.py

In [ ]:
import os, shutil
os.makedirs('/kaggle/working/PersonaLive/pretrained_weights', exist_ok=True)
src='/kaggle/working/Kemzy-LiveAvatar/backend/renderer/personalive_server.py'
dst='/kaggle/working/PersonaLive/personalive_server.py'
shutil.copy2(src,dst)
print('Kémzy server copied:', dst)
print('Diagnostic endpoint: POST /v1/diagnostics/render')

## Provide a real face image

Upload one clear portrait of a face. The notebook will use the same image as the reference and create four lightly transformed driving frames for the first smoke test. For the strongest proof, replace those four frames with four consecutive frames from a real camera/video.

In [ ]:
from IPython.display import display
from ipywidgets import FileUpload
u=FileUpload(accept='image/*', multiple=False)
display(u)

In [ ]:
from PIL import Image, ImageEnhance
import io, os
assert u.value, 'Upload a portrait in the widget above first.'
item=next(iter(u.value.values()))
raw=item['content']
img=Image.open(io.BytesIO(raw)).convert('RGB')
img.save('/kaggle/working/reference.jpg', quality=95)
# Four valid driving frames; real camera frames can replace these files later.
for i, factor in enumerate([0.98, 1.00, 1.02, 1.00], 1):
    frame=ImageEnhance.Brightness(img).enhance(factor)
    frame.save(f'/kaggle/working/frame{i}.jpg', quality=92)
print('Reference:', img.size)
print('Prepared 4 driving JPEG frames.')

In [ ]:
%cd /kaggle/working/PersonaLive
import subprocess, os, time
env=os.environ.copy()
env['MODEL_DIR']='/kaggle/working/PersonaLive'
env['PERSONALIVE_CONFIG']='/kaggle/working/PersonaLive/configs/prompts/personalive_online.yaml'
env['ACCELERATION']='none'
env['DIAGNOSTIC_RENDER_TIMEOUT']='180'
server=subprocess.Popen(['python','-m','uvicorn','personalive_server:app','--host','0.0.0.0','--port','7860'],env=env,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True)
   
time.sleep(8)
print('server pid:', server.pid)
print(subprocess.run(['curl','-s','http://127.0.0.1:7860/health'],capture_output=True,text=True).stdout)

In [ ]:
import requests, base64, json, os
files={
 'reference':('reference.jpg',open('/kaggle/working/reference.jpg','rb'),'image/jpeg'),
 'frame1':('frame1.jpg',open('/kaggle/working/frame1.jpg','rb'),'image/jpeg'),
 'frame2':('frame2.jpg',open('/kaggle/working/frame2.jpg','rb'),'image/jpeg'),
 'frame3':('frame3.jpg',open('/kaggle/working/frame3.jpg','rb'),'image/jpeg'),
 'frame4':('frame4.jpg',open('/kaggle/working/frame4.jpg','rb'),'image/jpeg')
}
r=requests.post('http://127.0.0.1:7860/v1/diagnostics/render',files=files,timeout=240)
print('HTTP:',r.status_code)
print(r.text[:2000])
r.raise_for_status()
result=r.json()
jpeg=base64.b64decode(result['first_frame_jpeg_base64'])
open('/kaggle/working/k​​emzy_personalive_render.jpg','wb').write(jpeg)
print('Verified generated JPEG bytes:',len(jpeg))

In [ ]:
from IPython.display import display
display(Image.open('/kaggle/working/k​​emzy_personalive_render.jpg'))